
# SmolLM: Transformer Implementation from Scratch

## Objective
Implement a decoder-only causal language model using PyTorch.

## Architecture
1. Configuration
2. RMSNorm
3. Rotary Positional Embeddings (RoPE)
4. Grouped-Query Attention (GQA)
5. SwiGLU feed-forward network
6. Pre-normalized decoder block
7. Full causal language model with tied embeddings

## Validation strategy
- Validate tensor shapes at each stage.
- Compare mathematical operations with reference implementations.
- Check causal masking and numerical stability.
- Verify parameter count and tied weights.
- Later, load pretrained checkpoint weights and compare logits.


In [146]:

import json
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoConfig

SEED = 42
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.14.1+cpu
Device: cpu


In [147]:
assert isinstance(DEVICE, torch.device)
assert torch.get_num_threads() >= 1

print("Environment checks passed.")

Environment checks passed.


In [148]:

MODEL_ID = "HuggingFaceTB/SmolLM2-135M"

hf_config = AutoConfig.from_pretrained(MODEL_ID)

print("Model type:", hf_config.model_type)
print("Hidden size:", hf_config.hidden_size)
print("Layers:", hf_config.num_hidden_layers)
print("Attention heads:", hf_config.num_attention_heads)
print("KV heads:", hf_config.num_key_value_heads)
print("Intermediate size:", hf_config.intermediate_size)
print("Vocabulary size:", hf_config.vocab_size)
print("Tied embeddings:", hf_config.tie_word_embeddings)


Model type: llama
Hidden size: 576
Layers: 30
Attention heads: 9
KV heads: 3
Intermediate size: 1536
Vocabulary size: 49152
Tied embeddings: True


In [149]:

@dataclass
class SmolLMConfig:
    vocab_size: int
    hidden_size: int
    intermediate_size: int
    num_hidden_layers: int
    num_attention_heads: int
    num_key_value_heads: int
    max_position_embeddings: int
    rms_norm_eps: float
    rope_theta: float
    tie_word_embeddings: bool
    attention_bias: bool = False
    mlp_bias: bool = False
    attention_dropout: float = 0.0

    @property
    def head_dim(self) -> int:
        assert self.hidden_size % self.num_attention_heads == 0
        return self.hidden_size // self.num_attention_heads

    def validate(self):
        assert self.vocab_size > 0
        assert self.hidden_size > 0
        assert self.num_hidden_layers > 0
        assert self.num_attention_heads > 0
        assert self.num_key_value_heads > 0

        assert self.hidden_size % self.num_attention_heads == 0
        assert self.num_attention_heads % self.num_key_value_heads == 0

        assert self.head_dim % 2 == 0
        assert self.intermediate_size > 0
        assert self.max_position_embeddings > 0
        assert self.rms_norm_eps > 0
        assert self.rope_theta > 0


config = SmolLMConfig(
    vocab_size=hf_config.vocab_size,
    hidden_size=hf_config.hidden_size,
    intermediate_size=hf_config.intermediate_size,
    num_hidden_layers=hf_config.num_hidden_layers,
    num_attention_heads=hf_config.num_attention_heads,
    num_key_value_heads=hf_config.num_key_value_heads,
    max_position_embeddings=hf_config.max_position_embeddings,
    rope_theta=getattr(hf_config, "rope_theta", 100000.0),
    rms_norm_eps=hf_config.rms_norm_eps,
    tie_word_embeddings=hf_config.tie_word_embeddings,
    attention_bias=hf_config.attention_bias,
    mlp_bias=getattr(hf_config, "mlp_bias", False),
    attention_dropout=hf_config.attention_dropout,
)

config.validate()
print(config)
print("Head dimension:", config.head_dim)


SmolLMConfig(vocab_size=49152, hidden_size=576, intermediate_size=1536, num_hidden_layers=30, num_attention_heads=9, num_key_value_heads=3, max_position_embeddings=8192, rms_norm_eps=1e-05, rope_theta=100000.0, tie_word_embeddings=True, attention_bias=False, mlp_bias=False, attention_dropout=0.0)
Head dimension: 64


In [150]:
assert config.hidden_size == 576
assert config.num_hidden_layers == 30
assert config.num_attention_heads == 9
assert config.num_key_value_heads == 3
assert config.head_dim == 64

print(config.rope_theta)

print("A1 passed.")

100000.0
A1 passed.



## A2 — RMSNorm

RMSNorm normalizes the magnitude of each token's hidden-state vector.

For input shape `(batch_size, sequence_length, hidden_size)`, it computes the
root mean square across the final dimension, then applies a learnable weight.

We accumulate the squared values in float32 for numerical stability, then
return the normalized result in the original input dtype.

Analogy: RMSNorm is like an automatic volume controller for each token's
feature vector. It controls the overall magnitude while preserving the
relative proportions of its features, before applying learned scaling.


In [151]:

class RMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_dtype = x.dtype

        x = x.float()
        rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)

        return (x * rms * self.weight.float()).to(input_dtype)

In [152]:
class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim: int,
        max_position_embeddings: int,
        theta: float = 100_000.0,
    ):
        super().__init__()

        assert head_dim % 2 == 0

        self.head_dim = head_dim
        self.theta = theta

        # Each pair of features uses its own rotation frequency.
        inv_freq = 1.0 / (
            theta ** (
                torch.arange(0, head_dim, 2, dtype=torch.float32)
                / head_dim
            )
        )

        self.register_buffer("inv_freq", inv_freq, persistent=False)

        positions = torch.arange(
            max_position_embeddings, dtype=torch.float32
        )

        freqs = torch.outer(positions, inv_freq)

        # Repeat each frequency for its corresponding feature pair.
        emb = torch.cat((freqs, freqs), dim=-1)

        self.register_buffer(
            "cos_cached", emb.cos(), persistent=False
        )
        self.register_buffer(
            "sin_cached", emb.sin(), persistent=False
        )

    def forward(self, x: torch.Tensor, position_ids=None):
        # x shape: (batch, heads, sequence_length, head_dim)
        seq_len = x.shape[-2]

        if position_ids is None:
            cos = self.cos_cached[:seq_len]
            sin = self.sin_cached[:seq_len]
        else:
            cos = self.cos_cached[position_ids]
            sin = self.sin_cached[position_ids]

        return cos, sin


In [153]:

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    half = x.shape[-1] // 2
    x1 = x[..., :half]
    x2 = x[..., half:]

    return torch.cat((-x2, x1), dim=-1)



def apply_rope(q, k, cos, sin):
    # q: [batch, query_heads, seq_len, head_dim]
    # k: [batch, kv_heads, seq_len, head_dim]
    # cos/sin: [batch, seq_len, head_dim] or [seq_len, head_dim]

    if cos.ndim == 2:
        cos = cos.unsqueeze(0)
        sin = sin.unsqueeze(0)

    # [batch, seq_len, head_dim] -> [batch, 1, seq_len, head_dim]
    cos = cos.unsqueeze(1).to(device=q.device, dtype=q.dtype)
    sin = sin.unsqueeze(1).to(device=q.device, dtype=q.dtype)

    q_rotated = q * cos + rotate_half(q) * sin

    # Match key dtype/device separately for robustness.
    cos_k = cos.to(device=k.device, dtype=k.dtype)
    sin_k = sin.to(device=k.device, dtype=k.dtype)
    k_rotated = k * cos_k + rotate_half(k) * sin_k

    return q_rotated, k_rotated



In [154]:

class GroupedQueryAttention(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()

        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = config.head_dim
        self.num_kv_groups = self.num_heads // self.num_kv_heads
        self.attention_dropout = config.attention_dropout

        # Query has more heads than key and value.
        self.q_proj = nn.Linear(
            self.hidden_size,
            self.num_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.k_proj = nn.Linear(
            self.hidden_size,
            self.num_kv_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.v_proj = nn.Linear(
            self.hidden_size,
            self.num_kv_heads * self.head_dim,
            bias=config.attention_bias,
        )

        self.o_proj = nn.Linear(
            self.num_heads * self.head_dim,
            self.hidden_size,
            bias=config.attention_bias,
        )

        self.rotary_emb = RotaryEmbedding(
            head_dim=self.head_dim,
            max_position_embeddings=config.max_position_embeddings,
            theta=config.rope_theta,
        )

    
    
    def forward(
        self,
        x: torch.Tensor,
        past_key_value=None,
        use_cache: bool = False,
        position_ids=None,
    ):
        batch_size, seq_len, _ = x.shape

        # 1. Project hidden states into Q, K, V.
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # 2. Reshape into attention heads.
        q = q.view(
            batch_size, seq_len, self.num_heads, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size, seq_len, self.num_kv_heads, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size, seq_len, self.num_kv_heads, self.head_dim
        ).transpose(1, 2)

        # 3. Apply RoPE using absolute token positions.
        cos, sin = self.rotary_emb(q, position_ids)
        q, k = apply_rope(q, k, cos, sin)

        # 4. Append current K/V to the existing cache.
        past_len = 0

        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(2)

            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)

        # 5. Preserve the compact KV cache: 3 heads, not 9.
        present_key_value = (k, v) if use_cache else None

        # 6. Expand KV heads for grouped-query attention.
        k_attn = k.repeat_interleave(
            self.num_kv_groups, dim=1
        )
        v_attn = v.repeat_interleave(
            self.num_kv_groups, dim=1
        )

        # 7. Apply causal masking correctly.
        # Full prefill: prevent attending to future tokens.
        # Cached decoding: new tokens can attend to all cached
        # tokens and earlier tokens in the current input.
        if past_len == 0:
            is_causal = True
            attn_mask = None
        else:
            key_len = k_attn.size(2)
            query_positions = (
                past_len
                + torch.arange(seq_len, device=x.device)
            )
            key_positions = torch.arange(
                key_len, device=x.device
            )

            attn_mask = (
                key_positions.unsqueeze(0)
                <= query_positions.unsqueeze(1)
            )
            is_causal = False

        # 8. Compute scaled dot-product attention.
        attn_output = F.scaled_dot_product_attention(
            q,
            k_attn,
            v_attn,
            attn_mask=attn_mask,
            dropout_p=(
                self.attention_dropout if self.training else 0.0
            ),
            is_causal=is_causal,
        )

        # 9. Merge heads and project back to hidden size.
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(
            batch_size, seq_len, self.hidden_size
        )

        output = self.o_proj(attn_output)

        if use_cache:
            return output, present_key_value

        return output


In [155]:
class SwiGLU(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()

        self.gate_proj = nn.Linear(
            config.hidden_size,
            config.intermediate_size,
            bias=config.mlp_bias,
        )

        self.up_proj = nn.Linear(
            config.hidden_size,
            config.intermediate_size,
            bias=config.mlp_bias,
        )

        self.down_proj = nn.Linear(
            config.intermediate_size,
            config.hidden_size,
            bias=config.mlp_bias,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = F.silu(self.gate_proj(x))
        up = self.up_proj(x)

        hidden = gate * up

        return self.down_proj(hidden)


In [156]:
class DecoderBlock(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()

        self.input_layernorm = RMSNorm(
            config.hidden_size,
            eps=config.rms_norm_eps,
        )

        self.self_attn = GroupedQueryAttention(config)

        self.post_attention_layernorm = RMSNorm(
            config.hidden_size,
            eps=config.rms_norm_eps,
        )

        self.mlp = SwiGLU(config)

    
    def forward(
        self,
        x: torch.Tensor,
        past_key_value=None,
        use_cache: bool = False,
        position_ids=None,
    ):
        if use_cache:
            attn_output, present_key_value = self.self_attn(
                self.input_layernorm(x),
                past_key_value=past_key_value,
                use_cache=True,
                position_ids=position_ids,
            )
        else:
            attn_output = self.self_attn(
                self.input_layernorm(x),
                position_ids=position_ids,
            )

        x = x + attn_output
        x = x + self.mlp(self.post_attention_layernorm(x))

        if use_cache:
            return x, present_key_value

        return x


In [157]:

class SmolLMForCausalLM(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()

        self.config = config

        # 1. Convert token IDs into hidden vectors
        self.embed_tokens = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
        )

        # 2. Stack independent Transformer blocks
        self.layers = nn.ModuleList([
                DecoderBlock(config)
            for _ in range(config.num_hidden_layers)
        ])

        # 3. Normalize the final hidden representations
        self.norm = RMSNorm(
            config.hidden_size,
            eps=config.rms_norm_eps,
        )

        # 4. Project hidden vectors to vocabulary logits
        self.lm_head = nn.Linear(
            config.hidden_size,
            config.vocab_size,
            bias=False,
        )

        # 5. Share input embedding and output projection weights
        if config.tie_word_embeddings:
            self.lm_head.weight = self.embed_tokens.weight

    
    def forward(
        self,
        input_ids: torch.Tensor,
        past_key_values=None,
        use_cache: bool = False,
    ):
        batch_size, seq_len = input_ids.shape

        # Each layer has its own cache. Determine the absolute position
        # of the first token in this input.
        past_length = 0
        if past_key_values is not None:
            past_length = past_key_values[0][0].shape[2]

        position_ids = torch.arange(
            past_length,
            past_length + seq_len,
            device=input_ids.device,
        ).unsqueeze(0).expand(batch_size, -1)

        x = self.embed_tokens(input_ids)
        next_cache = [] if use_cache else None

        for layer_idx, layer in enumerate(self.layers):
            past_key_value = (
                past_key_values[layer_idx]
                if past_key_values is not None
                else None
            )

            if use_cache:
                x, present_key_value = layer(
                    x,
                    past_key_value=past_key_value,
                    use_cache=True,
                    position_ids=position_ids,
                )
                next_cache.append(present_key_value)
            else:
                x = layer(x, position_ids=position_ids)

        x = self.norm(x)
        logits = self.lm_head(x)

        if use_cache:
            return logits, tuple(next_cache)

        return logits



In [158]:
from transformers import AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM2-135M"

hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
)
hf_model.eval()

print(type(hf_model).__name__)
print("Reference parameters:", sum(p.numel() for p in hf_model.parameters()))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

LlamaForCausalLM
Reference parameters: 134515008


In [159]:

def load_hf_weights(
    model: SmolLMForCausalLM,
    hf_model: nn.Module,
):
    source_state = hf_model.state_dict()
    target_state = model.state_dict()

    mapped_state = {}
    used_source_keys = set()
    ignored_source_keys = set()

    for source_key, tensor in source_state.items():
        # Our RoPE computes these buffers locally.
        if source_key.endswith(
            ("rotary_emb.inv_freq",
             "rotary_emb.cos_cached",
             "rotary_emb.sin_cached")
        ):
            ignored_source_keys.add(source_key)
            continue

        if source_key.startswith("model.layers."):
            target_key = source_key.removeprefix("model.")
        elif source_key in (
            "model.embed_tokens.weight",
            "model.norm.weight",
        ):
            target_key = source_key.removeprefix("model.")
        elif source_key.startswith("lm_head."):
            target_key = source_key
        else:
            raise KeyError(
                f"Unrecognized HF parameter: {source_key}"
            )

        if target_key not in target_state:
            raise KeyError(
                f"No destination for {source_key} -> {target_key}"
            )

        if target_key in mapped_state:
            raise KeyError(f"Duplicate mapping: {target_key}")

        if tensor.shape != target_state[target_key].shape:
            raise ValueError(
                f"Shape mismatch for {target_key}: "
                f"HF={tuple(tensor.shape)}, "
                f"ours={tuple(target_state[target_key].shape)}"
            )

        mapped_state[target_key] = tensor
        used_source_keys.add(source_key)

    # If the checkpoint omits a tied output-head entry,
    # reuse its input embedding tensor.
    if model.config.tie_word_embeddings:
        embedding_key = "embed_tokens.weight"
        head_key = "lm_head.weight"

        if head_key not in mapped_state:
            mapped_state[head_key] = mapped_state[embedding_key]

    missing_keys = set(target_state) - set(mapped_state)
    if missing_keys:
        raise KeyError(f"Missing destination weights: {sorted(missing_keys)}")

    unexpected_keys = (
        set(source_state) - used_source_keys - ignored_source_keys
    )
    if unexpected_keys:
        raise KeyError(f"Unused HF weights: {sorted(unexpected_keys)}")

    model.load_state_dict(mapped_state, strict=True)

    # Verify that tied weights still share the same parameter.
    if model.config.tie_word_embeddings:
        assert (
            model.lm_head.weight is model.embed_tokens.weight
        )

    print(f"Mapped tensors: {len(used_source_keys)}")
    print(f"Ignored local RoPE buffers: {len(ignored_source_keys)}")
    print("All destination tensors loaded successfully.")


In [160]:
model = SmolLMForCausalLM(config)
load_hf_weights(model, hf_model)

model.eval()

Mapped tensors: 273
Ignored local RoPE buffers: 0
All destination tensors loaded successfully.


SmolLMForCausalLM(
  (embed_tokens): Embedding(49152, 576)
  (layers): ModuleList(
    (0-29): 30 x DecoderBlock(
      (input_layernorm): RMSNorm()
      (self_attn): GroupedQueryAttention(
        (q_proj): Linear(in_features=576, out_features=576, bias=False)
        (k_proj): Linear(in_features=576, out_features=192, bias=False)
        (v_proj): Linear(in_features=576, out_features=192, bias=False)
        (o_proj): Linear(in_features=576, out_features=576, bias=False)
        (rotary_emb): RotaryEmbedding()
      )
      (post_attention_layernorm): RMSNorm()
      (mlp): SwiGLU(
        (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
        (up_proj): Linear(in_features=576, out_features=1536, bias=False)
        (down_proj): Linear(in_features=1536, out_features=576, bias=False)
      )
    )
  )
  (norm): RMSNorm()
  (lm_head): Linear(in_features=576, out_features=49152, bias=False)
)

In [161]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Vocabulary size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)
print("Special tokens:", tokenizer.special_tokens_map)

Vocabulary size: 49152
Model max length: 8192
Special tokens: {'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}


In [162]:
text = "Transformers learn useful representations of language."

input_ids = tokenizer.encode(
    text,
    add_special_tokens=False,
)

decoded_text = tokenizer.decode(input_ids)

print("Original:", text)
print("Token IDs:", input_ids)
print("Decoded:", decoded_text)

Original: Transformers learn useful representations of language.
Token IDs: [31806, 366, 835, 3465, 11605, 282, 1789, 30]
Decoded: Transformers learn useful representations of language.


In [163]:
assert decoded_text == text
print("Tokenizer round trip passed.")

Tokenizer round trip passed.


In [164]:
inputs = tokenizer(
    text,
    return_tensors="pt",
    add_special_tokens=False,
)

input_ids = inputs["input_ids"]

print(input_ids.shape)

with torch.no_grad():
    logits = model(input_ids)

print(logits.shape)

torch.Size([1, 8])
torch.Size([1, 8, 49152])


In [165]:

@torch.inference_mode()
def generate_greedy(
    model,
    input_ids,
    max_new_tokens=30,
    eos_token_id=None,
):
    model.eval()
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        output = model(generated)

        # Support both custom models and Hugging Face models
        logits = output.logits if hasattr(output, "logits") else output

        next_token = logits[:, -1, :].argmax(
            dim=-1, keepdim=True
        )

        generated = torch.cat([generated, next_token], dim=-1)

        if eos_token_id is not None:
            if (next_token == eos_token_id).all():
                break

    return generated


In [166]:

# prompt = "The capital of France is"

# inputs = tokenizer(prompt, return_tensors="pt")
# input_ids = inputs["input_ids"].to(device)

# custom_ids = generate_greedy(
#     model,
#     input_ids,
#     max_new_tokens=30,
#     eos_token_id=tokenizer.eos_token_id,
# )

# hf_ids = generate_greedy(
#     hf_model,
#     input_ids,
#     max_new_tokens=30,
#     eos_token_id=tokenizer.eos_token_id,
# )

# print("Custom model:", tokenizer.decode(custom_ids[0], skip_special_tokens=True))
# print("Hugging Face:", tokenizer.decode(hf_ids[0], skip_special_tokens=True))
# print("Exact token match:", torch.equal(custom_ids, hf_ids))


In [167]:
def apply_temperature(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    """Scale logits by a strictly positive temperature."""
    if temperature <= 0:
        raise ValueError("temperature must be greater than 0")

    return logits / temperature


In [168]:
def apply_top_k(logits: torch.Tensor, k: int) -> torch.Tensor:
    """Keep the k highest logits and mask all remaining logits."""
    vocab_size = logits.shape[-1]

    if not 1 <= k <= vocab_size:
        raise ValueError(f"k must be between 1 and {vocab_size}")

    top_k_values, _ = torch.topk(logits, k, dim=-1)
    threshold = top_k_values[..., -1, None]

    return logits.masked_fill(logits < threshold, float("-inf"))


In [169]:
def apply_top_p(logits: torch.Tensor, top_p: float) -> torch.Tensor:
    """Keep the smallest high-probability token set reaching top_p."""
    if not 0.0 < top_p <= 1.0:
        raise ValueError("top_p must be greater than 0 and at most 1")

    sorted_logits, sorted_indices = torch.sort(
        logits, descending=True, dim=-1
    )

    sorted_probs = torch.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Initially remove tokens whose cumulative probability exceeds top_p.
    sorted_remove = cumulative_probs > top_p

    # Keep the token that crosses the threshold as well.
    sorted_remove[..., 1:] = sorted_remove[..., :-1].clone()
    sorted_remove[..., 0] = False

    # Restore the original vocabulary ordering.
    remove_mask = torch.zeros_like(sorted_remove)
    remove_mask.scatter_(-1, sorted_indices, sorted_remove)

    return logits.masked_fill(remove_mask, float("-inf"))


In [170]:
print("Temperature scaling:")
prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]

device = next(model.parameters()).device
input_ids = input_ids.to(device)

model.eval()

with torch.inference_mode():
    logits = model(input_ids)[:, -1, :]

for temperature in (0.5, 1.0, 2.0):
    scaled_logits = apply_temperature(logits, temperature)
    probabilities = torch.softmax(scaled_logits, dim=-1)

    top_probs, top_ids = probabilities[0].topk(5)

    print(f"\nTemperature: {temperature}")
    for token_id, probability in zip(top_ids.tolist(), top_probs.tolist()):
        token = tokenizer.decode([token_id])
        print(f"  {token!r:20} {probability:.6f}")

print("\nTop-k sampling:")
prompt = "The capital of France is"
device = next(model.parameters()).device

input_ids = tokenizer(
    prompt, return_tensors="pt"
)["input_ids"].to(device)

model.eval()

with torch.inference_mode():
    logits = model(input_ids)[:, -1, :]

k = 5
filtered_logits = apply_top_k(logits, k)
probabilities = torch.softmax(filtered_logits, dim=-1)

print("Tokens remaining:", torch.isfinite(filtered_logits).sum().item())
print("Probability sum:", probabilities.sum().item())

top_probs, top_ids = probabilities[0].topk(k)

for token_id, probability in zip(top_ids.tolist(), top_probs.tolist()):
    print(f"{tokenizer.decode([token_id])!r:20} {probability:.6f}")


print("\nTop-p sampling:")
prompt = "The capital of France is"
device = next(model.parameters()).device

input_ids = tokenizer(
    prompt, return_tensors="pt"
)["input_ids"].to(device)

model.eval()

with torch.inference_mode():
    logits = model(input_ids)[:, -1, :]

top_p = 0.9
filtered_logits = apply_top_p(logits, top_p)
probabilities = torch.softmax(filtered_logits, dim=-1)

print("Tokens retained:", torch.isfinite(filtered_logits).sum().item())
print("Probability sum:", probabilities.sum().item())

top_probs, top_ids = probabilities[0].topk(5)

for token_id, probability in zip(top_ids.tolist(), top_probs.tolist()):
    print(f"{tokenizer.decode([token_id])!r:20} {probability:.6f}")




Temperature scaling:

Temperature: 0.5
  ' the'               0.771823
  ' Paris'             0.099259
  ' located'           0.060194
  ' called'            0.021693
  ' a'                 0.017330

Temperature: 1.0
  ' the'               0.261682
  ' Paris'             0.093842
  ' located'           0.073079
  ' called'            0.043871
  ' a'                 0.039211

Temperature: 2.0
  ' the'               0.012274
  ' Paris'             0.007350
  ' located'           0.006486
  ' called'            0.005026
  ' a'                 0.004751

Top-k sampling:
Tokens remaining: 5
Probability sum: 1.0
' the'               0.511412
' Paris'             0.183399
' located'           0.142820
' called'            0.085738
' a'                 0.076631

Top-p sampling:
Tokens retained: 406
Probability sum: 1.0
' the'               0.290708
' Paris'             0.104252
' located'           0.081185
' called'            0.048737
' a'                 0.043560


In [171]:

import torch

@torch.inference_mode()
def generate_sample(
    model,
    input_ids: torch.Tensor,
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: int | None = None,
    top_p: float | None = None,
    eos_token_id: int | None = None,
    seed: int | None = None,
) -> torch.Tensor:
    """Generate tokens using temperature-scaled, filtered sampling."""
    if temperature <= 0:
        raise ValueError("temperature must be greater than 0")
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens cannot be negative")

    device = input_ids.device
    generator = None

    if seed is not None:
        generator = torch.Generator(device=device)
        generator.manual_seed(seed)

    model.eval()
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        logits = model(generated)[:, -1, :]
        logits = apply_temperature(logits, temperature)

        if top_k is not None:
            logits = apply_top_k(logits, top_k)

        if top_p is not None:
            logits = apply_top_p(logits, top_p)

        probabilities = torch.softmax(logits, dim=-1)

        next_token = torch.multinomial(
            probabilities,
            num_samples=1,
            generator=generator,
        )

        generated = torch.cat([generated, next_token], dim=1)

        if eos_token_id is not None:
            if (next_token == eos_token_id).all():
                break

    return generated


In [172]:

print("Model class:", type(model).__name__)
print("Model forward:", type(model).forward)

print("\nFirst layer class:", type(model.layers[0]).__name__)
print("Layer forward:", type(model.layers[0]).forward)

print("\nAttention class:", type(model.layers[0].self_attn).__name__)
print("Attention forward:", type(model.layers[0].self_attn).forward)



import inspect

for name, module in [
    ("model", model),
    ("first layer", model.layers[0]),
    ("attention", model.layers[0].self_attn),
]:
    print(f"\n{name}: {type(module).__name__}")
    print(inspect.signature(type(module).forward))




Model class: SmolLMForCausalLM
Model forward: <function SmolLMForCausalLM.forward at 0x000002396C6CCFE0>

First layer class: DecoderBlock
Layer forward: <function DecoderBlock.forward at 0x000002396C6CD580>

Attention class: GroupedQueryAttention
Attention forward: <function GroupedQueryAttention.forward at 0x000002396C6CEB60>

model: SmolLMForCausalLM
(self, input_ids: torch.Tensor, past_key_values=None, use_cache: bool = False)

first layer: DecoderBlock
(self, x: torch.Tensor, past_key_value=None, use_cache: bool = False, position_ids=None)

attention: GroupedQueryAttention
(self, x: torch.Tensor, past_key_value=None, use_cache: bool = False, position_ids=None)


In [173]:

prompt = "The capital of France is"
device = next(model.parameters()).device

input_ids = tokenizer(
    prompt, return_tensors="pt"
)["input_ids"].to(device)

generation_args = {
    "max_new_tokens": 30,
    "temperature": 0.8,
    "top_k": 50,
    "top_p": 0.9,
    "eos_token_id": tokenizer.eos_token_id,
    "seed": 42,
}

sample_a = generate_sample(model, input_ids, **generation_args)
sample_b = generate_sample(model, input_ids, **generation_args)

print("Same token sequence:", torch.equal(sample_a, sample_b))
print("\nGenerated text:")
print(tokenizer.decode(sample_a[0], skip_special_tokens=True))


Same token sequence: True

Generated text:
The capital of France is Paris.

The Capital of France is Paris.

The Capital of France is Paris.

The Capital of France is Paris.



In [174]:

prompt = "The capital of France is"
device = next(model.parameters()).device

input_ids = tokenizer(
    prompt, return_tensors="pt"
)["input_ids"].to(device)

model.eval()

with torch.inference_mode():
    # Baseline: process the entire prompt.
    full_logits = model(input_ids)

    # Cached prefill: process the prompt and save K/V.
    prefill_logits, past_key_values = model(
        input_ids,
        use_cache=True,
    )

    # Feed the last prompt token again from the cached prefix,
    # so compare against the same complete context.
    prefix_ids = input_ids[:, :-1]
    last_token = input_ids[:, -1:]

    _, prefix_cache = model(
        prefix_ids,
        use_cache=True,
    )

    cached_last_logits, _ = model(
        last_token,
        past_key_values=prefix_cache,
        use_cache=True,
    )

difference = (
    full_logits[:, -1, :] - cached_last_logits[:, -1, :]
).abs()

print("Maximum logit difference:", difference.max().item())
print("Mean logit difference:", difference.mean().item())

print(
    "Same next-token prediction:",
    full_logits[:, -1, :].argmax(dim=-1).item()
    == cached_last_logits[:, -1, :].argmax(dim=-1).item()
)

print("Cache layers:", len(prefix_cache))
print("First-layer cached K shape:", tuple(prefix_cache[0][0].shape))
print("First-layer cached V shape:", tuple(prefix_cache[0][1].shape))


Maximum logit difference: 2.288818359375e-05
Mean logit difference: 2.6311988676752662e-06
Same next-token prediction: True
Cache layers: 30
First-layer cached K shape: (1, 3, 4, 64)
First-layer cached V shape: (1, 3, 4, 64)


In [175]:

@torch.inference_mode()
def generate_sample_cached(
    model,
    input_ids: torch.Tensor,
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: int | None = None,
    top_p: float | None = None,
    eos_token_id: int | None = None,
    seed: int | None = None,
) -> torch.Tensor:
    """Sample tokens while reusing each layer's KV cache."""
    if temperature <= 0:
        raise ValueError("temperature must be greater than 0")
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens cannot be negative")

    device = input_ids.device
    generator = None

    if seed is not None:
        generator = torch.Generator(device=device)
        generator.manual_seed(seed)

    model.eval()
    generated = input_ids.clone()

    if max_new_tokens == 0:
        return generated

    # Prefill: process the entire prompt once.
    logits, past_key_values = model(
        input_ids,
        use_cache=True,
    )

    for step in range(max_new_tokens):
        next_logits = logits[:, -1, :]
        next_logits = apply_temperature(next_logits, temperature)

        if top_k is not None:
            next_logits = apply_top_k(next_logits, top_k)

        if top_p is not None:
            next_logits = apply_top_p(next_logits, top_p)

        probabilities = torch.softmax(next_logits, dim=-1)

        next_token = torch.multinomial(
            probabilities,
            num_samples=1,
            generator=generator,
        )

        generated = torch.cat([generated, next_token], dim=1)

        if eos_token_id is not None and (next_token == eos_token_id).all():
            break

        # The final sampled token needs no forward pass if generation ends.
        if step == max_new_tokens - 1:
            break

        # Decode only the newly sampled token, reusing the K/V cache.
        logits, past_key_values = model(
            next_token,
            past_key_values=past_key_values,
            use_cache=True,
        )

    return generated


In [176]:

prompt = "The capital of France is"
device = next(model.parameters()).device

input_ids = tokenizer(
    prompt, return_tensors="pt"
)["input_ids"].to(device)

generation_args = {
    "max_new_tokens": 30,
    "temperature": 0.8,
    "top_k": 50,
    "top_p": 0.9,
    "eos_token_id": tokenizer.eos_token_id,
    "seed": 42,
}

uncached_output = generate_sample(
    model, input_ids, **generation_args
)

cached_output = generate_sample_cached(
    model, input_ids, **generation_args
)

print(
    "Exact token match:",
    torch.equal(uncached_output, cached_output),
)

print("\nUncached:")
print(tokenizer.decode(uncached_output[0], skip_special_tokens=True))

print("\nCached:")
print(tokenizer.decode(cached_output[0], skip_special_tokens=True))


Exact token match: True

Uncached:
The capital of France is Paris.

The Capital of France is Paris.

The Capital of France is Paris.

The Capital of France is Paris.


Cached:
The capital of France is Paris.

The Capital of France is Paris.

The Capital of France is Paris.

The Capital of France is Paris.



In [88]:
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb

custom_block = model.layers[0]
hf_block = hf_model.model.layers[0]

prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)

with torch.inference_mode():
    hidden = hf_model.model.embed_tokens(input_ids)
    hidden = hf_block.input_layernorm(hidden)

    custom_attn = custom_block.self_attn
    hf_attn = hf_block.self_attn

    batch_size, seq_len, _ = hidden.shape

    q = custom_attn.q_proj(hidden).view(
        batch_size, seq_len,
        custom_attn.num_heads, custom_attn.head_dim
    ).transpose(1, 2)

    k = custom_attn.k_proj(hidden).view(
        batch_size, seq_len,
        custom_attn.num_kv_heads, custom_attn.head_dim
    ).transpose(1, 2)

    position_ids = torch.arange(
        seq_len, device=device
    ).unsqueeze(0)

    custom_cos, custom_sin = custom_attn.rotary_emb(q)

    # Your Transformers version computes RoPE at the model level.
    hf_cos, hf_sin = hf_model.model.rotary_emb(hidden, position_ids)

    custom_q_rot, custom_k_rot = apply_rope(
        q, k, custom_cos, custom_sin
    )

    hf_q_rot, hf_k_rot = apply_rotary_pos_emb(
        q, k, hf_cos, hf_sin
    )

    for name, a, b in [
        ("RoPE cosine", custom_cos, hf_cos),
        ("RoPE sine", custom_sin, hf_sin),
        ("Rotated Q", custom_q_rot, hf_q_rot),
        ("Rotated K", custom_k_rot, hf_k_rot),
    ]:
        diff = (a.float() - b.float()).abs()
        print(
            f"{name:15s} max={diff.max().item():.8f}, "
            f"mean={diff.mean().item():.8f}"
        )

RoPE cosine     max=0.00000000, mean=0.00000000
RoPE sine       max=0.00000000, mean=0.00000000
Rotated Q       max=0.00000000, mean=0.00000000
Rotated K       max=0.00000000, mean=0.00000000


In [188]:
hf_block = hf_model.model.layers[0]

print("Decoder block:", type(hf_block).__name__)
print("Attention module:", type(hf_block.self_attn).__name__)
print("Attention attributes:")
print([name for name in vars(hf_block.self_attn) if "rotary" in name.lower()])
print("Model rotary attributes:")
print([name for name in vars(hf_model.model) if "rotary" in name.lower()])

Decoder block: LlamaDecoderLayer
Attention module: LlamaAttention
Attention attributes:
[]
Model rotary attributes:
[]


In [90]:
import inspect

print("Decoder forward:")
print(inspect.signature(hf_model.model.layers[0].forward))

print("\nModel forward:")
print(inspect.signature(hf_model.model.forward))

print("\nRoPE-related modules:")
for name, module in hf_model.named_modules():
    if "rotary" in name.lower() or "rotary" in type(module).__name__.lower():
        print(name, type(module).__name__)

Decoder forward:
(hidden_states: torch.Tensor, attention_mask: torch.Tensor | None = None, position_ids: torch.LongTensor | None = None, past_key_values: transformers.cache_utils.Cache | None = None, use_cache: bool | None = False, position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None, **kwargs: *<class 'transformers.utils.generic.TransformersKwargs'>) -> torch.Tensor

Model forward:
(input_ids: torch.LongTensor | None = None, attention_mask: torch.Tensor | None = None, position_ids: torch.LongTensor | None = None, past_key_values: transformers.cache_utils.Cache | None = None, inputs_embeds: torch.FloatTensor | None = None, use_cache: bool | None = None, **kwargs: *<class 'transformers.utils.generic.TransformersKwargs'>) -> transformers.modeling_outputs.BaseModelOutputWithPast

RoPE-related modules:
model.rotary_emb LlamaRotaryEmbedding


In [91]:
hf_config_dict = hf_model.config.to_dict()

print("RoPE-related HF config values:")
for key, value in hf_config_dict.items():
    if "rope" in key.lower() or "position" in key.lower():
        print(f"{key}: {value}")

print("\nHF rotary embedding:")
print(hf_model.model.rotary_emb)

print("\nHF inv_freq (first 8):")
print(hf_model.model.rotary_emb.inv_freq[:8])

print("\nCustom rotary embedding:")
custom_rope = model.layers[0].self_attn.rotary_emb
print("theta:", custom_rope.theta)
print("inv_freq (first 8):")
print(custom_rope.inv_freq[:8])

RoPE-related HF config values:
max_position_embeddings: 8192
rope_parameters: {'rope_theta': 100000, 'rope_type': 'default'}
rope_interleaved: False

HF rotary embedding:
LlamaRotaryEmbedding()

HF inv_freq (first 8):
tensor([1.0000, 0.6978, 0.4870, 0.3398, 0.2371, 0.1655, 0.1155, 0.0806])

Custom rotary embedding:
theta: 100000.0
inv_freq (first 8):
tensor([1.0000, 0.6978, 0.4870, 0.3398, 0.2371, 0.1655, 0.1155, 0.0806])


In [179]:
import torch

ck = torch.load("./checkpoints_smollm2/model.pt", map_location="cpu", weights_only=False)
print(type(ck))

if isinstance(ck, dict):
    for k, v in ck.items():
        if isinstance(v, dict):
            print(f"{k}: dict with {len(v)} keys, e.g. {list(v.keys())[:3]}")
        elif hasattr(v, "shape"):
            print(f"{k}: {type(v).__name__} shape={tuple(v.shape)} dtype={getattr(v, 'dtype', None)}")
        else:
            print(f"{k}: {type(v).__name__}")

<class 'dict'>
model: dict with 273 keys, e.g. ['embed_tokens.weight', 'layers.0.input_layernorm.weight', 'layers.0.self_attn.q_proj.weight']
optimizer: dict with 2 keys, e.g. ['state', 'param_groups']
scheduler: dict with 7 keys, e.g. ['base_lrs', 'last_epoch', '_step_count']
scaler: dict with 0 keys, e.g. []
global_step: int
epoch: int
batch_cursor: int
epoch_permutation: NoneType
shuffle_generator: Tensor shape=(5056,) dtype=torch.uint8
torch_rng: Tensor shape=(5056,) dtype=torch.uint8
numpy_rng: tuple
python_rng: tuple
cuda_rng: list
history: dict with 7 keys, e.g. ['train_steps', 'train_loss', 'val_steps']
baseline: dict with 3 keys, e.g. ['loss', 'perplexity', 'tokens']
precision: str
block_size: int
train_batch_size: int
grad_accum_steps: int
num_epochs: int


In [ ]:
sd = ck["model_state_dict"]
for i, (k, v) in enumerate(sd.items()):
    if i < 15:
        print(k, tuple(v.shape), v.dtype)
print("total tensors:", len(sd))

KeyError: 'model_state_dict'

In [181]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ck = torch.load("./checkpoints_smollm2/model.pt", map_location="cpu", weights_only=False)
sd = ck["model"]
print("precision:", ck["precision"], "| epoch:", ck["epoch"], "| step:", ck["global_step"])
print("non-layer keys:", [k for k in sd if not k.startswith("layers.")])
print("dtype:", next(iter(sd.values())).dtype)

# HF skeleton with the original SmolLM2 config
hf = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M", torch_dtype=torch.float32)

# add "model." prefix; lm_head stays top-level
new_sd = {(k if k.startswith("lm_head") else "model." + k): v.float() for k, v in sd.items()}

missing, unexpected = hf.load_state_dict(new_sd, strict=False)
print("missing:", missing)
print("unexpected:", unexpected)

# if lm_head is in your checkpoint, confirm it is tied to the embedding
if "lm_head.weight" in sd:
    print("lm_head tied to embed:", torch.equal(sd["lm_head.weight"], sd["embed_tokens.weight"]))

precision: bf16 | epoch: 3 | step: 1341
non-layer keys: ['embed_tokens.weight', 'norm.weight', 'lm_head.weight']
dtype: torch.float32


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

missing: []
unexpected: []
lm_head tied to embed: True


In [185]:
hf.eval()
tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
prompt = "..."  # use one of the same prompts you used in the notebook
out = hf.generate(**tok(prompt, return_tensors="pt"), max_new_tokens=60, do_sample=False)
print(tok.decode(out[0]))

hf.save_pretrained("smollm2-trained", safe_serialization=True)
tok.save_pretrained("smollm2-trained")

...

Task: Find every Category that can be reached from a Customer in two steps, first through REVIEWED and then through IN_CATEGORY.

```cypher
MATCH (a:Customer {name: $name})-[:REVIEWED]->(:


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('smollm2-trained\\tokenizer_config.json', 'smollm2-trained\\tokenizer.json')

In [187]:
!python llama.cpp/convert_hf_to_gguf.py ./smollm2-trained --outtype f16 --outfile ./smollm2-trained-f16.gguf

INFO:hf-to-gguf:Loading model: smollm2-trained
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.float32 --> F16, shape = {576, 49152}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float32 --> F32, shape = {576}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float32 --> F16, shape = {1536, 576}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float32 --> F16, shape = {576, 1536}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float32 --> F16, shape = {576, 1536}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float32 --> F32, shape = {576}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float32 --> F16, shape = {576, 192}
INFO:hf-to-gguf:blk.0.attn_output.weight,    torch.float32 --> F16, shape = {576, 576}
INFO:hf-to-gguf:blk.0.attn_q.weight,       